# M10 OBSO + C-OBSO — run partido a partido (resumable)

Procesa los 64 partidos WC22 a **25 Hz full quality**, uno a uno.

Cachea cada partido por separado en `data/parquet/derived/offball/per_match/{match_id}.parquet`.
Se puede **interrumpir y retomar**: al re-ejecutar el NB salta los partidos ya cacheados.

Cuando los 64 esten cacheados, las celdas finales agregan `per_minute.parquet` y `per_shock_window.parquet`.

In [ ]:
# Setup
import sys, gc, time
from pathlib import Path
import polars as pl

_REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(_REPO / 'src'))

from M01_loader_pff import list_event_match_ids
import M10_offball as m10

PER_MATCH_DIR = _REPO / 'data' / 'parquet' / 'derived' / 'offball' / 'per_match'
PER_MATCH_DIR.mkdir(parents=True, exist_ok=True)
print(f'cache dir: {PER_MATCH_DIR}')

In [ ]:
# xG grid (cached, ~5s)
xg_grid = m10.build_xg_grid(cache=True)
print(f'xG grid {xg_grid.shape}: min={xg_grid.min():.4f} max={xg_grid.max():.4f}')

In [ ]:
# Loop partido a partido. Salta los ya cacheados.
# Tiempo por partido a 25 Hz: ~10-15 min.  64 partidos = ~12-15h total.
# Interrumpir es seguro: re-ejecutar el NB retoma desde donde quedo.

match_ids = list_event_match_ids()
done   = {int(p.stem) for p in PER_MATCH_DIR.glob('*.parquet')}
todo   = [m for m in match_ids if m not in done]
print(f'progreso: {len(done)}/{len(match_ids)} cacheados, {len(todo)} pendientes')

for i, mid in enumerate(todo):
    t0 = time.time()
    try:
        df = m10.compute_obso_match(mid, xg_grid)
    except Exception as e:
        print(f'  [{i+1}/{len(todo)}] match {mid} FAIL: {e}')
        continue
    out = PER_MATCH_DIR / f'{mid}.parquet'
    df.write_parquet(out, compression='snappy', statistics=True)
    elapsed = time.time() - t0
    total_done = len(done) + i + 1
    print(f'  [{total_done}/{len(match_ids)}] match {mid}: {df.height:>5} rows en {elapsed:>6.1f}s -> {out.name}', flush=True)
    del df
    gc.collect()

print(f'\ncompletados: {len(list(PER_MATCH_DIR.glob("*.parquet")))}/{len(match_ids)}')

## Agregacion final

Solo correr cuando los 64 partidos esten cacheados. Concatena los 64 parquets y aplica la agregacion shock-window.

In [ ]:
# Concat de los 64 per-match -> per_minute.parquet
files = sorted(PER_MATCH_DIR.glob('*.parquet'))
assert len(files) == 64, f'faltan {64 - len(files)} partidos por procesar'

big = pl.concat([pl.read_parquet(f) for f in files])
out_pm = _REPO / 'data' / 'parquet' / 'derived' / 'offball' / 'per_minute.parquet'
big.write_parquet(out_pm, compression='snappy', statistics=True)
print(f'per_minute: {big.height:,} filas, {big["pff_match_id"].n_unique()} matches -> {out_pm}')
print(big.head(3))

In [ ]:
# Per-shock-window aggregation usando la funcion de M10
import importlib; importlib.reload(m10)
per_shock = m10.aggregate_per_shock_window(cache=False)
out_ps = _REPO / 'data' / 'parquet' / 'derived' / 'offball' / 'per_shock_window.parquet'
per_shock.write_parquet(out_ps, compression='snappy', statistics=True)
print(f'per_shock_window: {per_shock.height} filas -> {out_ps}')

# Sanity delta por shock_type
summary = per_shock.group_by('shock_type').agg([
    pl.col('obso_pre').mean().alias('obso_pre'),
    pl.col('obso_post').mean().alias('obso_post'),
    (pl.col('obso_post') - pl.col('obso_pre')).mean().alias('delta_obso'),
    pl.col('c_obso_pre').mean().alias('c_obso_pre'),
    pl.col('c_obso_post').mean().alias('c_obso_post'),
])
print(summary)